In [1]:
import pandas as pd
import numpy as np

In [2]:
from src.backend.recommendation import recommend_from_profile

In [3]:
profile_test = {
    "type_formation": "BUT",
    "is_apprentissage": True,
    "max_frais_scolarite": 2000,
    "commune": None,  # ou une commune si tu veux
}

profile_test

{'type_formation': 'BUT',
 'is_apprentissage': True,
 'max_frais_scolarite': 2000,
 'commune': None}

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")
question = "Je veux un BUT informatique en apprentissage pas trop cher"
query_emb = model.encode(question, normalize_embeddings=True)
query_emb.shape

/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA/env_etudia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7829.38it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(768,)

# Test de `reason` pour voir si le profil match à la formation

In [5]:
results = recommend_from_profile(profile_test, query_emb, limit=3)
len(results), results[0]

(3,
 {'chunk_id': '16349_0',
  'chunk_text': 'Nom de la formation : BUT - Génie électrique et informatique industrielle - en apprentissage\n\nType de formation : BUT\n\nÉtablissement : IUT GRAND OUEST NORMANDIE - Pôle de Cherbourg - Site de Cherbourg-en-Cotentin (50), Publics (Cherbourg-en-Cotentin, Manche, Normandie)\n\nMentions / spécialités : BUT - Automatisme et Informatique Industrielle,BUT - Génie électrique et informatique industrielle\n\nSélection : formation sélective',
  'type_formation': 'BUT',
  'commune': 'Cherbourg-en-Cotentin',
  'frais_scolarite': None,
  'distance': 0.1381764155082731,
  'reason': {'match_type_formation': True,
   'match_apprentissage': True,
   'under_budget': True},
  'explanation': "correspond au type de formation que tu recherches ; propose de l'apprentissage, comme tu le souhaites ; respecte ton budget (ou ne dépasse pas le plafond indiqué)"})

In [6]:
results[0]["reason"]

{'match_type_formation': True,
 'match_apprentissage': True,
 'under_budget': True}

In [7]:
results = recommend_from_profile(profile_test, query_emb, limit=3)
results[0]["explanation"]

"correspond au type de formation que tu recherches ; propose de l'apprentissage, comme tu le souhaites ; respecte ton budget (ou ne dépasse pas le plafond indiqué)"

In [8]:
from src.backend.profile_schema import StudentProfile

profile_test: StudentProfile = {
    "type_formation": "BUT",
    "is_apprentissage": True,
    "max_frais_scolarite": 2000,
    "commune": None,
    "domains_interet": ["sport", "maths"],
    "matieres_aimees": ["maths"],
    "matieres_evitees": ["histoire-géo"],
    "preference_rythme": "matin",
    "preference_travail": "équipe",
    "niveau_idee_orientation": "aucune_idee",
}

results = recommend_from_profile(profile_test, query_emb, limit=3)
results[0]["reason"], results[0]["explanation"]

({'match_type_formation': True,
  'match_apprentissage': True,
  'under_budget': True},
 "correspond au type de formation que tu recherches ; propose de l'apprentissage, comme tu le souhaites ; respecte ton budget (ou ne dépasse pas le plafond indiqué)")

# Test d'un profil fictif

In [9]:
from src.backend.profile_from_text import infer_profile_from_text
from src.backend.recommendation import recommend_from_profile
from sentence_transformers import SentenceTransformer

message = (
    "J'aime le sport, les maths et pas l'histoire-géo. "
    "Je fais du tennis en club et j'aime travailler en équipe. "
    "Je n'ai aucune idée vers quoi me diriger."
)

# 1) Embedding de la question
model = SentenceTransformer("intfloat/multilingual-e5-base")
query_emb = model.encode(message, normalize_embeddings=True)

# 2) Profil structuré (plus tard via LLM, pour l'instant NotImplementedError)
# profile = infer_profile_from_text(message)

# 3) Exemple de profil simulé pour tester la suite
profile_simule = {
    "type_formation": "BUT",
    "is_apprentissage": True,
    "max_frais_scolarite": 2000,
    "domains_interet": ["sport", "maths"],
    "matieres_aimees": ["maths"],
    "matieres_evitees": ["histoire-géo"],
    "preference_rythme": "matin",
    "preference_travail": "équipe",
    "niveau_idee_orientation": "aucune_idee",
}

results = recommend_from_profile(profile_simule, query_emb, limit=3)
results[0]["reason"], results[0].get("explanation")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14955.50it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


({'match_type_formation': True,
  'match_apprentissage': True,
  'under_budget': True},
 "correspond au type de formation que tu recherches ; propose de l'apprentissage, comme tu le souhaites ; respecte ton budget (ou ne dépasse pas le plafond indiqué)")

# Test de la fonction à vide

In [10]:
from src.backend.chat_pipeline import orienter_lyceen

message_test = (
    "J'aime le sport et les maths, je fais du tennis en club, "
    "et j'aime travailler en équipe. Je ne sais pas vers quoi aller."
)

try:
    reponse = orienter_lyceen(message_test)
except NotImplementedError as e:
    print("Pipeline logique en place, LLM à brancher plus tard :", e)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6843.21it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline logique en place, LLM à brancher plus tard : LLM-based profile inference not implemented yet.
